## Double gyre flow tracking test case

In [1]:
# IMPORT UTILITIES

import random
import numpy as np
import torch

random.seed(0)
np.random.seed(0)
torch.manual_seed(0);

In [2]:
# PROBLEM DEFINITION

from utils.doublegyre import DoubleGyre

doublegyre = DoubleGyre(Lx = 2.0, Ly = 1.0, nx = 100, ny = 50, mu = 0.01, gamma = 1.0, dt = 0.1, T = 5.0)

In [ ]:
# SOLUTION EXAMPLE

vortex_coords = (1.0, 0.5)
amplitude_ref = 0.25
frequency_ref = np.pi

[vt, pt, ut, vt_ref] = doublegyre.solve_NS(vortex_coords, control = True, amplitude_ref = amplitude_ref, frequency_ref = frequency_ref)

In [ ]:
# VELOCITY VISUALIZATION

from utils.postprocessing import trajectories

trajectories([vt_ref[:-1], vt[:-1], ut],
             doublegyre.plot_velocity,
             titles = ['Reference flow', 'Controlled flow', 'Control'],
             times = doublegyre.times,
             figsize = (15, 5))

In [ ]:
# VORTICITY VISUALIZATION

trajectories([vt_ref[:-1], vt[:-1], ut],
             doublegyre.plot_vorticity,
             titles = ['Reference flow', 'Controlled flow', 'Control'],
             times = doublegyre.times,
             figsize = (15, 5))

## Data loading

In [ ]:
# DATA GENERATION (SKIP THIS CELL IF DATA ALREADY AVAILABLE)

ntrajectories = 100
vortex_coords = (1.0, 0.5)
parameters_ranges = [(0.0, 0.5), (np.pi/4, 2*np.pi)] # Parameters: double gyre flow amplitude and frequency
nparameters = len(parameters_ranges)

V = np.empty((ntrajectories, doublegyre.ntimesteps, doublegyre.nvelocity))
P = np.empty((ntrajectories, doublegyre.ntimesteps, doublegyre.npressure))
U = np.empty((ntrajectories, doublegyre.ntimesteps-1, doublegyre.nvelocity))
V_ref = np.empty((ntrajectories, doublegyre.ntimesteps, doublegyre.nvelocity))
MU = np.empty((ntrajectories, doublegyre.ntimesteps, nparameters))

for i in range(ntrajectories):

    print(f"Generating snapshots n.{i}...")

    # generate random parameters
    parameters = [0] * nparameters
    for j in range(len(parameters_ranges)):
        parameters[j] = (parameters_ranges[j][1] - parameters_ranges[j][0]) * np.random.rand() + parameters_ranges[j][0]    
    
    # compute trajectory
    amplitude_ref = parameters[0]
    frequency_ref = parameters[1]
    [vt, pt, ut, vt_ref] = doublegyre.solve_NS(vortex_coords, control = True, amplitude_ref = amplitude_ref, frequency_ref = frequency_ref)

    # save results
    V[i] = vt
    P[i] = pt
    U[i] = ut
    V_ref[i] = vt_ref
    MU[i] = parameters
    np.savez("data/DoubleGyre/doublegyre.npz", velocity = V, pressure = P, control = U, reference = V_ref, parameters = MU)

print("Snapshots generated!")

In [4]:
# LOAD DATA

from utils.datamanager import DataManager

vortex_coords = (1.0, 0.5)
parameters_ranges = [(0.0, 0.5), (np.pi/4, 2*np.pi)] # Parameters: double gyre flow amplitude and frequency
nparameters = len(parameters_ranges)

datamanager = DataManager(np.load("data/DoubleGyre/doublegyre.npz"),
                          dims = {'velocity': 2, 'pressure': 1, 'control': 2, 'reference': 2},
                          train_ratio = 0.8,
                          valid_ratio = 0.1,
                          test_ratio = 0.1)

In [ ]:
# VORTICITY VISUALIZATION

which_trajectory = 0

trajectories([datamanager.data["reference"][which_trajectory], datamanager.data["velocity"][which_trajectory], datamanager.data["control"][which_trajectory]],
             doublegyre.plot_vorticity,
             titles = ['Reference flow', 'Controlled flow', 'Control'],
             times = doublegyre.times,
             figsize = (15, 5))

In [5]:
# TRAIN-VALIDATION-TEST SPLITTING

datamanager.prepare()

## Data compression

In [ ]:
# PROPER ORTHOGONAL DECOMPOSITION (POD)

kvelocity = kcontrol = 30
datamanager.POD(ranks = {'velocity': kvelocity, 'control': kcontrol},
                starting_times = {'velocity': 1, 'control': 1})

## SHRED-ROM

In [ ]:
# EXTRACT SENSORS DATA (SKIP THIS CELL IF DATA ALREADY AVAILABLE)

nsensors = 3
sensors_pressure = np.random.choice(doublegyre.npressure, size = nsensors, replace = False)
sensors_velocity = np.random.choice(doublegyre.nvelocity // 2, size = nsensors, replace = False)
sensors_coordinates_pressure = torch.from_numpy(doublegyre.Ph.tabulate_dof_coordinates()[sensors_pressure]).reshape(-1)
sensors_coordinates_velocity = torch.from_numpy(doublegyre.Vh.sub(0).collapse().tabulate_dof_coordinates()[sensors_velocity]).reshape(-1)

sensors_data_train = np.concatenate((datamanager.data["pressure"][datamanager.train][:,:-1,sensors_pressure],
                                datamanager.data["reference"][:,:,0 : doublegyre.nvelocity : 2][datamanager.train][:,:-1,sensors_velocity]), 2)
sensors_data_valid = np.concatenate((datamanager.data["pressure"][datamanager.valid][:,:-1,sensors_pressure],
                                datamanager.data["reference"][:,:,0 : doublegyre.nvelocity : 2][datamanager.valid][:,:-1,sensors_velocity]), 2)
sensors_data_test = np.concatenate((datamanager.data["pressure"][datamanager.test][:,:-1,sensors_pressure],
                                datamanager.data["reference"][:,:,0 : doublegyre.nvelocity : 2][datamanager.test][:,:-1,sensors_velocity]), 2)


In [7]:
# LOAD SENSORS DATA

nsensors = 3
sensors_pressure = torch.load('data/DoubleGyre/sensors_pressure.pt', weights_only = False)
sensors_velocity = torch.load('data/DoubleGyre/sensors_velocity.pt', weights_only = False)
sensors_coordinates_pressure = torch.from_numpy(doublegyre.Ph.tabulate_dof_coordinates()[sensors_pressure]).reshape(-1)
sensors_coordinates_velocity = torch.from_numpy(doublegyre.Vh.sub(0).collapse().tabulate_dof_coordinates()[sensors_velocity]).reshape(-1)

sensors_data_train = np.concatenate((datamanager.data["pressure"][datamanager.train][:,:-1,sensors_pressure],
                                datamanager.data["reference"][:,:,0 : doublegyre.nvelocity : 2][datamanager.train][:,:-1,sensors_velocity]), 2)
sensors_data_valid = np.concatenate((datamanager.data["pressure"][datamanager.valid][:,:-1,sensors_pressure],
                                datamanager.data["reference"][:,:,0 : doublegyre.nvelocity : 2][datamanager.valid][:,:-1,sensors_velocity]), 2)
sensors_data_test = np.concatenate((datamanager.data["pressure"][datamanager.test][:,:-1,sensors_pressure],
                                datamanager.data["reference"][:,:,0 : doublegyre.nvelocity : 2][datamanager.test][:,:-1,sensors_velocity]), 2)


In [8]:
# BUILD TRAIN, VALIDATION AND TEST DATASETS WITH PADDING

from utils.datamanager import Padding, TimeSeriesDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

lag = 25

train_data_in = Padding(torch.from_numpy(sensors_data_train).float(), lag).to(device)
valid_data_in = Padding(torch.from_numpy(sensors_data_valid).float(), lag).to(device)
test_data_in = Padding(torch.from_numpy(sensors_data_test).float(), lag).to(device)

train_data_out = Padding(torch.from_numpy(np.concatenate((datamanager.data_POD["control"][datamanager.train,:,:],
                                                          datamanager.data_POD["velocity"][datamanager.train,:,:]), 2)) ,1).squeeze(1).to(device)
valid_data_out = Padding(torch.from_numpy(np.concatenate((datamanager.data_POD["control"][datamanager.valid,:,:],
                                                          datamanager.data_POD["velocity"][datamanager.valid,:,:]), 2)) ,1).squeeze(1).to(device)
test_data_out = Padding(torch.from_numpy(np.concatenate((datamanager.data_POD["control"][datamanager.test,:,:],
                                                          datamanager.data_POD["velocity"][datamanager.test,:,:]), 2)) ,1).squeeze(1).to(device)

train_dataset = TimeSeriesDataset(train_data_in, train_data_out)
valid_dataset = TimeSeriesDataset(valid_data_in, valid_data_out)
test_dataset = TimeSeriesDataset(test_data_in, test_data_out)

In [ ]:
# SHRED TRAINING (SKIP THIS CELL IF MODEL ALREADY AVAILABLE)

from utils.models import SHRED, fit

nlatent = 64
shred = SHRED(2*nsensors, kcontrol + kvelocity, hidden_size = nlatent, hidden_layers = 2, decoder_sizes = [350, 400], dropout = 0.1).to(device)
train_errors, valid_errors = fit(shred, train_dataset, valid_dataset, batch_size = 64, epochs = 500, lr = 1e-3, verbose = True, patience = 100)
train_errors, valid_errors = fit(shred, train_dataset, valid_dataset, batch_size = 64, epochs = 500, lr = 1e-4, verbose = True, patience = 100)

In [9]:
# LOAD PRE-TRAINED SHRED MODEL

from utils.models import SHRED

nlatent = 64
shred = SHRED(2*nsensors, kcontrol + kvelocity, hidden_size = nlatent, hidden_layers = 2, decoder_sizes = [350, 400], dropout = 0.1).to(device)
shred.load_state_dict(torch.load('data/DoubleGyre/shred.pt', weights_only = False, map_location = torch.device('cpu')));

In [ ]:
# SHRED ERRORS ON TEST DATA

from utils.postprocessing import mre_numpy, num2p

shred.freeze()

shred_pred = shred(test_data_in).cpu().numpy()
shred_pred = shred_pred.reshape(datamanager.ntest, -1, kcontrol + kvelocity)

data_test_pred = datamanager.decode(data_POD = {'control': shred_pred[:,:,:kcontrol], 'velocity': shred_pred[:,:,kcontrol:]})

print(f"MRE on control: {num2p(mre_numpy(datamanager.data['control'][datamanager.test, 1:], data_test_pred['control']))}")
print(f"MRE on velocity: {num2p(mre_numpy(datamanager.data['velocity'][datamanager.test,1:], data_test_pred['velocity']))}")

In [ ]:
# GROUND TRUTH vs SHRED-ROM PREDICTION

which_trajectory = 0

trajectories([datamanager.data["velocity"][datamanager.test][which_trajectory, 1:],
              data_test_pred['velocity'][which_trajectory],
              np.sqrt((datamanager.data["velocity"][datamanager.test][which_trajectory, 1:] - data_test_pred['velocity'][which_trajectory])**2)],
              doublegyre.plot_vorticity,
              titles = ['Ground truth', 'SHRED-ROM prediction', 'Prediction error'],
              times = doublegyre.times[1:],
              figsize = (15, 5))

In [ ]:
# GROUND TRUTH vs SHRED-ROM PREDICTION

which_trajectory = 0

trajectories([datamanager.data["control"][datamanager.test][which_trajectory, 1:],
              data_test_pred['control'][which_trajectory],
              np.sqrt((datamanager.data["control"][datamanager.test][which_trajectory, 1:] - data_test_pred["control"][which_trajectory])**2)],
              doublegyre.plot_vorticity,
              titles = ['Ground truth', 'SHRED-ROM prediction', 'Prediction error'],
              times = doublegyre.times[1:],
              figsize = (15, 5))

## Latent sensor forecaster

In [ ]:
# LOAD SENSORS DATA

nsensors = 3
sensors_pressure = torch.load('data/DoubleGyre/sensors_pressure.pt', weights_only = False)
sensors_velocity = torch.load('data/DoubleGyre/sensors_velocity.pt', weights_only = False)
sensors_coordinates_pressure = torch.from_numpy(doublegyre.Ph.tabulate_dof_coordinates()[sensors_pressure]).reshape(-1)
sensors_coordinates_velocity = torch.from_numpy(doublegyre.Vh.sub(0).collapse().tabulate_dof_coordinates()[sensors_velocity]).reshape(-1)

sensors_data_train = np.concatenate((datamanager.data["pressure"][datamanager.train][:,:-1,sensors_pressure],
                                datamanager.data["reference"][:,:,0 : doublegyre.nvelocity : 2][datamanager.train][:,:-1,sensors_velocity]), 2)
sensors_data_valid = np.concatenate((datamanager.data["pressure"][datamanager.valid][:,:-1,sensors_pressure],
                                datamanager.data["reference"][:,:,0 : doublegyre.nvelocity : 2][datamanager.valid][:,:-1,sensors_velocity]), 2)
sensors_data_test = np.concatenate((datamanager.data["pressure"][datamanager.test][:,:-1,sensors_pressure],
                                datamanager.data["reference"][:,:,0 : doublegyre.nvelocity : 2][datamanager.test][:,:-1,sensors_velocity]), 2)

In [ ]:
# LOAD PRE-TRAINED SHRED MODEL

from utils.models import SHRED

nlatent = 64
shred = SHRED(2*nsensors, kcontrol + kvelocity, hidden_size = nlatent, hidden_layers = 2, decoder_sizes = [350, 400], dropout = 0.1).to(device)
shred.load_state_dict(torch.load('data/DoubleGyre/shred.pt', weights_only = False, map_location = torch.device('cpu')));
shred.freeze()

In [11]:
# BUILD TRAIN, VALIDATION AND TEST DATASETS WITH PADDING

from utils.datamanager import Padding, TimeSeriesDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

lag = 25

z_train = shred.sensor2latent(train_data_in).reshape(datamanager.ntrain, -1, nlatent)[:,:-1,:]
z_valid = shred.sensor2latent(valid_data_in).reshape(datamanager.nvalid, -1, nlatent)[:,:-1,:]
z_test = shred.sensor2latent(test_data_in).reshape(datamanager.ntest, -1, nlatent)[:,:-1,:]

train_data_in_forecaster = Padding(torch.cat((torch.from_numpy(sensors_data_train[:,:-1,:]), z_train), 2), lag).to(device)
valid_data_in_forecaster = Padding(torch.cat((torch.from_numpy(sensors_data_valid[:,:-1,:]), z_valid), 2), lag).to(device)
test_data_in_forecaster = Padding(torch.cat((torch.from_numpy(sensors_data_test[:,:-1,:]), z_test), 2), lag).to(device)

train_data_out_forecaster = Padding(torch.from_numpy(sensors_data_train[:,1:,:]), 1).squeeze(1).to(device)
valid_data_out_forecaster = Padding(torch.from_numpy(sensors_data_valid[:,1:,:]), 1).squeeze(1).to(device)
test_data_out_forecaster = Padding(torch.from_numpy(sensors_data_test[:,1:,:]), 1).squeeze(1).to(device)

train_dataset_forecaster = TimeSeriesDataset(train_data_in_forecaster, train_data_out_forecaster)
valid_dataset_forecaster = TimeSeriesDataset(valid_data_in_forecaster, valid_data_out_forecaster)
test_dataset_forecaster = TimeSeriesDataset(test_data_in_forecaster, test_data_out_forecaster)

In [ ]:
# LATENT SENSOR FORECASTER TRAINING

from utils.models import LatentSensorForecaster, fit

latentsensorforecaster = LatentSensorForecaster(2*nsensors + nlatent, 2*nsensors, hidden_size = 64, hidden_layers = 2, dropout = 0.1).to(device)
train_errors, valid_errors = fit(latentsensorforecaster, train_dataset_forecaster, valid_dataset_forecaster, batch_size = 64, epochs = 500, lr = 1e-3, verbose = True, patience = 100)
train_errors, valid_errors = fit(latentsensorforecaster, train_dataset_forecaster, valid_dataset_forecaster, batch_size = 64, epochs = 500, lr = 1e-4, verbose = True, patience = 100)

In [12]:
# LOAD PRE-TRAINED LATENT SENSOR FORECASTER

from utils.models import LatentSensorForecaster

latentsensorforecaster = LatentSensorForecaster(2*nsensors + nlatent, 2*nsensors, hidden_size = 64, hidden_layers = 2, dropout = 0.1).to(device)
latentsensorforecaster.load_state_dict(torch.load('data/DoubleGyre/latentsensorforecaster.pt', weights_only = False, map_location = torch.device('cpu')));

In [ ]:
# LATENT SENSOR FORECASTER ERRORS ON TEST DATA

from utils.postprocessing import mre_numpy, num2p

latentsensorforecaster.freeze()

sensor_test_hat = latentsensorforecaster(test_data_in_forecaster).cpu().numpy().reshape(datamanager.ntest, -1, 2*nsensors)

print(f"Mean relative latent sensor forecaster prediction error: {num2p(mre_numpy(sensors_data_test[:,1:,:], sensor_test_hat))}")

## SHRED-ROM into play

In [ ]:
# DOUBLE GYRE FLOW TRACKING FOR TEST PARAMETERS

from fenics import Function
import matplotlib.pyplot as plt
from IPython.display import clear_output as clc
from utils.postprocessing import mse_numpy

vortex_coords = (1.0, 0.5)
amplitude_ref_test = 0.25
frequency_ref_test = np.pi
vt_ref = doublegyre.double_gyre_flow(amplitude_ref_test, frequency_ref_test)
vt = [doublegyre.vortex(vortex_coords = vortex_coords).vector()[:]]
pt = np.zeros((1, doublegyre.npressure))
sensors_test = torch.zeros(doublegyre.ntimesteps + lag - 1, 2*nsensors)

Js = [mse_numpy(vt[0], vt_ref[0])]
for j in range(doublegyre.ntimesteps-1):
   
    # Extract sensor values
    sensors_test[j+lag-1] = torch.from_numpy(np.concatenate((pt[-1, sensors_pressure], vt_ref[j, 0 : doublegyre.nvelocity : 2][sensors_velocity]), 0))

    # Predict optimal control
    shred_pred = shred(sensors_test[j:j+lag,:].unsqueeze(0)).cpu().numpy()
    shred_pred = shred_pred.reshape(1, 1, kcontrol + kvelocity)
    data_test_pred = datamanager.decode(data_POD = {'control': shred_pred[:,:,:kcontrol], 'velocity': shred_pred[:,:,kcontrol:]})
        
    # Apply optimal control
    [vt, pt] = doublegyre.solve_NS(vortex_coords, v0_val = vt[-1], p0_val = pt[-1], control = True, ut_val = data_test_pred["control"][0], amplitude_ref = amplitude_ref_test, frequency_ref = frequency_ref_test, ntimesteps = 2)
    
    # Store result
    Js.append(mse_numpy(vt[-1], vt_ref[j+1]))

clc()
plt.plot(Js, 'magenta', linewidth = 2)
plt.xlabel("Time")
plt.ylabel("Loss function");

In [ ]:
# DOUBLE GYRE FLOW TRACKING FOR TEST PARAMETERS WITH MISSING SENSOR DATA

from fenics import Function
import matplotlib.pyplot as plt
from IPython.display import clear_output as clc
from utils.postprocessing import mse_numpy

vortex_coords = (1.0, 0.5)
amplitude_ref_test = 0.25
frequency_ref_test = np.pi
vt_ref = doublegyre.double_gyre_flow(amplitude_ref_test, frequency_ref_test)
vt = [doublegyre.vortex(vortex_coords = vortex_coords).vector()[:]]
pt = np.zeros((1, doublegyre.npressure))
sensors_test = torch.zeros(doublegyre.ntimesteps + lag - 1, 2*nsensors)
z_test = torch.zeros(doublegyre.ntimesteps + lag - 1, nlatent)

Js = [mse_numpy(vt[0], vt_ref[0])]
for j in range(doublegyre.ntimesteps-1):
   
    # Extract sensor values
    if j % 10 != 0 or j == 0: # Sensors ON
        sensors_test[j+lag-1] = torch.from_numpy(np.concatenate((pt[-1, sensors_pressure], vt_ref[j, 0 : doublegyre.nvelocity : 2][sensors_velocity]), 0))
    else: # Sensors OFF
        sensors_test[j+lag-1] = latentsensorforecaster(torch.cat((sensors_test[j-1:j+lag-1,:].unsqueeze(0), z_test[j-1:j+lag-1,:].unsqueeze(0)),2).cpu())[0]
    z_test[j+lag-1] = shred.sensor2latent(sensors_test[j:j+lag,:].unsqueeze(0)).cpu()

    # Predict optimal control
    shred_pred = shred(sensors_test[j:j+lag,:].unsqueeze(0)).cpu().numpy()
    shred_pred = shred_pred.reshape(1, 1, kcontrol + kvelocity)
    data_test_pred = datamanager.decode(data_POD = {'control': shred_pred[:,:,:kcontrol], 'velocity': shred_pred[:,:,kcontrol:]})
        
    # Apply optimal control
    [vt, pt] = doublegyre.solve_NS(vortex_coords, v0_val = vt[-1], p0_val = pt[-1], control = True, ut_val = data_test_pred["control"][0], amplitude_ref = amplitude_ref_test, frequency_ref = frequency_ref_test, ntimesteps = 2)

    # Store result
    Js.append(mse_numpy(vt[-1], vt_ref[j+1]))

clc()
plt.plot(Js, 'magenta', linewidth = 2)
plt.xlabel("Time")
plt.ylabel("Loss function");